# Homework #5: Model Deployment — F1 Podium Prediction

**Author:** yy3462  
**Course:** GR5069 — Applied Data Science

## Task
Build **two** predictive models on the F1 datasets, log each in MLflow, and write predictions from each model into **two separate tables** in my own Databricks database.

## Approach
- **Data:** Join `results`, `races`, `drivers`, `constructors`, and `qualifying`
- **Target:** Binary classification — podium finish (top 3)?
- **Model 1:** `RandomForestClassifier`
- **Model 2:** `GradientBoostingClassifier`
- **MLflow (per model):** hyperparameters, model, 4 metrics, 2 artifacts
- **Database:** `yy3462_db.rf_predictions` and `yy3462_db.gbt_predictions`

## 1. Imports

In [0]:
%pip install mlflow -q
dbutils.library.restartPython()

In [0]:
import os
import tempfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

import mlflow
import mlflow.sklearn

print("MLflow version:", mlflow.__version__)

## 2. Load F1 Data

In [0]:
BASE = "/Volumes/gr5069/raw/f1_data"

results_df      = spark.read.csv(f"{BASE}/results.csv",      header=True, inferSchema=True).toPandas()
races_df        = spark.read.csv(f"{BASE}/races.csv",        header=True, inferSchema=True).toPandas()
drivers_df      = spark.read.csv(f"{BASE}/drivers.csv",      header=True, inferSchema=True).toPandas()
constructors_df = spark.read.csv(f"{BASE}/constructors.csv", header=True, inferSchema=True).toPandas()
qualifying_df   = spark.read.csv(f"{BASE}/qualifying.csv",   header=True, inferSchema=True).toPandas()

print("results     :", results_df.shape)
print("races       :", races_df.shape)
print("drivers     :", drivers_df.shape)
print("constructors:", constructors_df.shape)
print("qualifying  :", qualifying_df.shape)

## 3. Feature Engineering & Joins

In [0]:
# Start from results
df = results_df[[
    "raceId", "driverId", "constructorId", "grid", "positionOrder", "points", "laps"
]].copy()

# Merge race metadata
df = df.merge(races_df[["raceId", "year", "round", "circuitId"]], on="raceId", how="left")

# Merge driver info -> driver age
drivers_small = drivers_df[["driverId", "dob", "nationality"]].rename(columns={"nationality": "driver_nationality"})
df = df.merge(drivers_small, on="driverId", how="left")
df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
df["driver_age"] = df["year"] - df["dob"].dt.year

# Merge constructor nationality
cons_small = constructors_df[["constructorId", "nationality"]].rename(columns={"nationality": "constructor_nationality"})
df = df.merge(cons_small, on="constructorId", how="left")

# Merge qualifying position
qual_small = qualifying_df.groupby(["raceId", "driverId"], as_index=False)["position"].min()
qual_small = qual_small.rename(columns={"position": "qual_position"})
df = df.merge(qual_small, on=["raceId", "driverId"], how="left")

print("Merged shape:", df.shape)

In [0]:
# Target: podium (top 3)
df["podium"] = (df["positionOrder"] <= 3).astype(int)

# Clean
df = df[df["grid"] > 0].copy()
df = df.dropna(subset=["driver_age"])
df["qual_position"] = df["qual_position"].fillna(99)

# Encode categoricals
df["driver_nat_code"]      = df["driver_nationality"].astype("category").cat.codes
df["constructor_nat_code"] = df["constructor_nationality"].astype("category").cat.codes

# Features
feature_cols = [
    "grid", "qual_position", "constructorId", "circuitId",
    "year", "round", "driver_age", "driver_nat_code", "constructor_nat_code"
]
X = df[feature_cols]
y = df["podium"]

# Keep IDs for prediction tables
df_ids = df[["raceId", "driverId", "constructorId"]].copy()

print("X shape:", X.shape)
print("Podium rate:", round(y.mean(), 3))

## 4. Train/Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Keep corresponding IDs for writing predictions later
ids_train, ids_test = train_test_split(
    df_ids, test_size=0.25, random_state=42, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)

## 5. Create Database and Prediction Tables

Creating two tables in my own database — one per model.

In [0]:
# Use my pre-assigned personal schema in gr5069 catalog
# (no CREATE SCHEMA needed — it already exists)

MY_SCHEMA = "gr5069.yy3462"

# Table 1: Random Forest predictions
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {MY_SCHEMA}.rf_predictions (
    raceId INT,
    driverId INT,
    constructorId INT,
    actual_podium INT,
    predicted_podium INT,
    predicted_proba DOUBLE,
    model_name STRING,
    run_id STRING
)
''')

# Table 2: Gradient Boosting predictions
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {MY_SCHEMA}.gbt_predictions (
    raceId INT,
    driverId INT,
    constructorId INT,
    actual_podium INT,
    predicted_podium INT,
    predicted_proba DOUBLE,
    model_name STRING,
    run_id STRING
)
''')

print(f"Tables created in {MY_SCHEMA}")
spark.sql(f"SHOW TABLES IN {MY_SCHEMA}").show()

## 6. MLflow Experiment Setup

In [0]:
EXPERIMENT_NAME = "/Users/yy3462@columbia.edu/hw5_f1_deployment"

try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

mlflow.set_experiment(EXPERIMENT_NAME)
print("Experiment ID:", experiment_id)

## 7. Logging Function

Reusable function: trains a model, logs to MLflow (hyperparameters, model, **4 metrics**, **2 artifacts**), returns predictions.

In [0]:
def train_and_log(experiment_id, run_name, model, params,
                  X_train, X_test, y_train, y_test, feature_names):
    with mlflow.start_run(experiment_id=experiment_id, run_name=run_name) as run:
        # Train
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        # Log model
        mlflow.sklearn.log_model(model, "model")

        # Log hyperparameters
        for p, v in params.items():
            mlflow.log_param(p, v)

        # Log 4 metrics
        metrics = {
            "accuracy":  accuracy_score(y_test, preds),
            "f1":        f1_score(y_test, preds, zero_division=0),
            "roc_auc":   roc_auc_score(y_test, proba),
            "precision": precision_score(y_test, preds, zero_division=0),
        }
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
            print(f"  {k:12s}: {v:.4f}")

        # Artifact 1: confusion matrix PNG
        cm = confusion_matrix(y_test, preds)
        fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["No Podium", "Podium"],
                    yticklabels=["No Podium", "Podium"], ax=ax_cm)
        ax_cm.set_xlabel("Predicted")
        ax_cm.set_ylabel("Actual")
        ax_cm.set_title(f"Confusion Matrix - {run_name}")
        plt.tight_layout()
        temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png", delete=False)
        try:
            fig_cm.savefig(temp.name)
            mlflow.log_artifact(temp.name, "confusion-matrix.png")
        finally:
            temp.close()
            os.unlink(temp.name)
        plt.close(fig_cm)

        # Artifact 2: ROC curve PNG
        fpr, tpr, _ = roc_curve(y_test, proba)
        fig_roc, ax_roc = plt.subplots(figsize=(5, 4))
        ax_roc.plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
        ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax_roc.set_xlabel("False Positive Rate")
        ax_roc.set_ylabel("True Positive Rate")
        ax_roc.set_title(f"ROC Curve - {run_name}")
        ax_roc.legend(loc="lower right")
        plt.tight_layout()
        temp = tempfile.NamedTemporaryFile(prefix="roc-curve-", suffix=".png", delete=False)
        try:
            fig_roc.savefig(temp.name)
            mlflow.log_artifact(temp.name, "roc-curve.png")
        finally:
            temp.close()
            os.unlink(temp.name)
        plt.close(fig_roc)

        run_id = run.info.run_id
        print(f"  -> run_id: {run_id}")
        return preds, proba, run_id, metrics

## 8. Model 1: Random Forest Classifier

In [0]:
rf_params = {
    "n_estimators": 300,
    "max_depth": 15,
    "class_weight": "balanced",
    "random_state": 42
}

rf_model = RandomForestClassifier(**rf_params)

rf_preds, rf_proba, rf_run_id, rf_metrics = train_and_log(
    experiment_id, "Model 1 - RandomForest", rf_model, rf_params,
    X_train, X_test, y_train, y_test, feature_names=feature_cols
)
print(f"\nRandom Forest run_id: {rf_run_id}")

## 9. Model 2: Gradient Boosting Classifier

In [0]:
gbt_params = {
    "n_estimators": 200,
    "max_depth": 5,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "random_state": 42
}

gbt_model = GradientBoostingClassifier(**gbt_params)

gbt_preds, gbt_proba, gbt_run_id, gbt_metrics = train_and_log(
    experiment_id, "Model 2 - GradientBoosting", gbt_model, gbt_params,
    X_train, X_test, y_train, y_test, feature_names=feature_cols
)
print(f"\nGradient Boosting run_id: {gbt_run_id}")

## 10. Compare the Two Models

In [0]:
comparison = pd.DataFrame({
    "Metric": ["accuracy", "f1", "roc_auc", "precision"],
    "RandomForest": [rf_metrics[m] for m in ["accuracy", "f1", "roc_auc", "precision"]],
    "GradientBoosting": [gbt_metrics[m] for m in ["accuracy", "f1", "roc_auc", "precision"]],
})
comparison["Winner"] = comparison.apply(
    lambda row: "RandomForest" if row["RandomForest"] >= row["GradientBoosting"]
                else "GradientBoosting", axis=1
)
comparison.round(4)

In [0]:
# Side-by-side bar chart
x = np.arange(len(comparison))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, comparison["RandomForest"],  width, label="RandomForest",      color="steelblue")
ax.bar(x + width/2, comparison["GradientBoosting"], width, label="GradientBoosting", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(comparison["Metric"])
ax.set_ylabel("Score")
ax.set_title("Model Comparison: RandomForest vs GradientBoosting")
ax.legend()
ax.set_ylim(0, 1.05)
for i, row in comparison.iterrows():
    ax.text(i - width/2, row["RandomForest"] + 0.02,  f'{row["RandomForest"]:.3f}',  ha="center", fontsize=9)
    ax.text(i + width/2, row["GradientBoosting"] + 0.02, f'{row["GradientBoosting"]:.3f}', ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 11. Write Predictions to Database Tables

For each model, build a DataFrame with IDs + actual + predicted + model metadata, then write to the corresponding table.

In [0]:
# Model 1: RF predictions -> gr5069.yy3462.rf_predictions
rf_pred_df = ids_test.copy()
rf_pred_df["actual_podium"]    = y_test.values
rf_pred_df["predicted_podium"] = rf_preds
rf_pred_df["predicted_proba"]  = rf_proba
rf_pred_df["model_name"]       = "RandomForestClassifier"
rf_pred_df["run_id"]           = rf_run_id

rf_spark_df = spark.createDataFrame(rf_pred_df)
rf_spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gr5069.yy3462.rf_predictions")

print(f"Wrote {rf_spark_df.count()} rows to gr5069.yy3462.rf_predictions")
rf_spark_df.show(5)

In [0]:
# Model 2: GBT predictions -> gr5069.yy3462.gbt_predictions
gbt_pred_df = ids_test.copy()
gbt_pred_df["actual_podium"]    = y_test.values
gbt_pred_df["predicted_podium"] = gbt_preds
gbt_pred_df["predicted_proba"]  = gbt_proba
gbt_pred_df["model_name"]       = "GradientBoostingClassifier"
gbt_pred_df["run_id"]           = gbt_run_id

gbt_spark_df = spark.createDataFrame(gbt_pred_df)
gbt_spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gr5069.yy3462.gbt_predictions")

print(f"Wrote {gbt_spark_df.count()} rows to gr5069.yy3462.gbt_predictions")
gbt_spark_df.show(5)

## 12. Verify Tables

In [0]:
print("=== gr5069.yy3462.rf_predictions ===")
spark.sql("SELECT COUNT(*) as row_count FROM gr5069.yy3462.rf_predictions").show()
spark.sql("SELECT * FROM gr5069.yy3462.rf_predictions LIMIT 5").show()

print("\n=== gr5069.yy3462.gbt_predictions ===")
spark.sql("SELECT COUNT(*) as row_count FROM gr5069.yy3462.gbt_predictions").show()
spark.sql("SELECT * FROM gr5069.yy3462.gbt_predictions LIMIT 5").show()

In [0]:
# Prediction distribution
print("Random Forest prediction distribution:")
spark.sql("SELECT predicted_podium, COUNT(*) as cnt FROM gr5069.yy3462.rf_predictions GROUP BY predicted_podium").show()

print("Gradient Boosting prediction distribution:")
spark.sql("SELECT predicted_podium, COUNT(*) as cnt FROM gr5069.yy3462.gbt_predictions GROUP BY predicted_podium").show()

## Summary

| Requirement | Implementation |
|---|---|
| **2 new tables** | `yy3462_db.rf_predictions` and `yy3462_db.gbt_predictions` — each stores raceId, driverId, constructorId, actual label, predicted label, predicted probability, model name, and MLflow run_id |
| **2 predictive models with MLflow** | Model 1 = RandomForestClassifier, Model 2 = GradientBoostingClassifier. Each logs hyperparameters, the model, 4 metrics (accuracy, f1, roc_auc, precision), and 2 artifacts (confusion matrix PNG, ROC curve PNG) |
| **Predictions stored in database** | RF predictions written to `yy3462_db.rf_predictions`; GBT predictions written to `yy3462_db.gbt_predictions` |
| **Code pushed to GitHub** | This notebook is committed to the HW5 GitHub Classroom repo |